# Module 0: Setup

> Self-directed module, ~10 min.

This notebook prepares the environment for the rest of the modules. We verify Python and dependencies, configure API keys, and confirm each external service is reachable.

Three outcomes by the end:

- Python, dependencies, and `.env` confirmed
- Model provider, Tavily, and LangSmith credentials verified
- Each service reachable end-to-end


## Module Map

| # | Notebook | Topic |
|---|----------|-------|
| 00 | *this notebook* | Environment setup and verification |
| 01 | `01_build_a_deep_agent_optional.ipynb` | Build a deep agent — **optional**, skip if already familiar with `deepagents` |
| 02 | `02_tracing.ipynb` | Tracing and trace queries |
| 03 | `03_finding_failure_modes.ipynb` | Finding failure modes (Chat, Insights Agent, Engine) |
| 04 | `04_datasets_and_experiments.ipynb` | Datasets and experiments (offline evaluations) |
| 05 | `05_online_evals.ipynb` | Online evaluations |
| 06 | `06_annotation_queues.ipynb` | Annotation queues |
| 07 | `07_deploy_and_govern_optional.ipynb` | Deploy + Govern — LangSmith Deployments and gateway policies — **optional**, requires deployment permissions |

Each module is self-contained but assumes the prior **required** modules have run. The optional modules (01, 07) can be skipped without breaking the rest.

## Skipping Module 01

Module 01 builds a deep agent from scratch. Those already familiar with the `deepagents` framework — custom tools, subagents, prompts — can skip directly to Module 02. The completed agent lives at `agents/partner_growth_agent.py` and is imported by Modules 02–06 via `utils/config.py`.


## Step 1 — System Prerequisites

Python 3.11+ and a small set of LangChain / LangSmith libraries are required. The first cell installs them via `uv sync` (skip if you've already synced). The second cell confirms the Python version and that the key libraries import cleanly, and defines two helpers (`_ok` / `_fail`) used by the rest of the notebook to report verification results.


In [ ]:
!uv sync


In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


def _ok(label, detail=""):
    print(f"\u2713 {label}{(' \u2014 ' + detail) if detail else ''}")


def _fail(label, exc):
    print(f"\u2717 {label}: {exc}")


assert sys.version_info >= (3, 11), f"Python 3.11+ required, found {sys.version}"
_ok(f"Python {sys.version_info.major}.{sys.version_info.minor}")

for pkg in ["langchain", "langgraph", "langsmith", "deepagents", "tavily", "openai"]:
    try:
        __import__(pkg)
        _ok(f"{pkg} importable")
    except ImportError as e:
        _fail(pkg, e)

Each verification prints a `✓` on success. If any line shows `✗`, the dependency or version check failed — install or upgrade before continuing with Step 2.


## Step 2 — API Keys

Three external services are used across the modules. The cell below copies `.env.example` to `.env` at the repo root; populate the values after it runs.

### Create your .env file


In [ ]:
!cp .env.example .env


| Key | Required for | Where to obtain |
|-----|--------------|-----------------|
| `ANTHROPIC_API_KEY` | Default model provider | <https://console.anthropic.com> |
| `LANGSMITH_API_KEY` | Tracing and evaluations | <https://smith.langchain.com> (see below) |
| `TAVILY_API_KEY` | Web search tool used by the agent | <https://tavily.com> |

### Using a different model provider

**Anthropic** is the default. **OpenAI**, **Azure OpenAI**, and **AWS Bedrock** are also supported — uncomment the corresponding initializer in `utils/models.py` and set the matching variables in `.env`. See `.env.example` for the full set of supported provider variables.

### Accepting an organization invitation

Accept the invitation email from LangSmith and complete sign-up using your email address and a password.

### Creating a LangSmith API key

1. Sign in at <https://smith.langchain.com>.
2. Navigate to **Settings → Organizations → Access and Security → API Keys** and click **+ API Key**.
3. Select the key type:
   - **Personal Access Token** (`lsv2_pt_...`) — for individual development. Sufficient for Modules 02–05.
   - **Service Key** (`lsv2_sk_...`) — for servers, CI, and deployments. Recommended for Module 06.
4. Set the expiration (or `Never`).
5. Click **Create API Key**.
6. Copy the key immediately — LangSmith displays it only once.

<img src="../images/create_api_key.png" alt="LangSmith Settings — API Keys" style="width: auto; max-height: 360px; border-radius: 8px;">

### Self-hosted LangSmith

Organizations running LangSmith self-hosted should set `LANGSMITH_ENDPOINT` in `.env` to the internal URL. The default is `https://api.smith.langchain.com`.


In [ ]:
import os
from dotenv import load_dotenv

# Load .env from the repo root and report which keys are present.
# Missing keys aren't prompted for — the verification cells below will fail
# at the point where each key is actually used (Step 3 / 4 / 5).
load_dotenv(project_root / ".env", override=True)

# Report whichever model-provider key is set (only one is required, depending on utils/models.py).
for var in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "AZURE_OPENAI_API_KEY", "AWS_ACCESS_KEY_ID"]:
    if os.environ.get(var):
        _ok(var, "set")

# These two are required regardless of model provider.
for var in ["LANGSMITH_API_KEY", "TAVILY_API_KEY"]:
    if os.environ.get(var):
        _ok(var, "set")
    else:
        _fail(var, "missing — set in .env before continuing")

endpoint = os.environ.get("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
flavor = "cloud" if "smith.langchain.com" in endpoint else "self-hosted"
_ok(f"LANGSMITH_ENDPOINT = {endpoint}", flavor)


## Step 3 — Verify LLM API Key

A small chat completion confirms the configured model is reachable and the API key is valid. `utils/models.py` is the single point of model initialization — by default Anthropic's `claude-sonnet-4-6`. The file also contains commented-out initializers for OpenAI, Azure, and AWS Bedrock; uncomment whichever provider you intend to use.


In [ ]:
from utils.models import model

try:
    response = model.invoke("Reply with the single word 'pong'.")
    _ok("LLM", f"model replied: {response.content.strip()[:40]}")
except Exception as e:
    _fail("LLM", e)

## Step 4 — Verify Tavily

The agent uses Tavily for web search. A single query confirms the key.

`utils/search.py` wraps Tavily with retries and a topic-matched canned fallback so the modules complete if Tavily is briefly unavailable. If the cell below reports `fell back to canned content`, the key is invalid or Tavily is unreachable — resolve before continuing; otherwise live traces in the evaluation modules will reflect canned responses rather than real search results.


In [ ]:
from utils.search import resilient_tavily_search

try:
    result = resilient_tavily_search("LangSmith observability", max_retries=1)
    if result.startswith("[fallback"):
        _fail("Tavily", "fell back to canned content — check your TAVILY_API_KEY")
    else:
        first_line = result.splitlines()[0] if result else "(empty)"
        _ok("Tavily", f"first hit: {first_line[:60]}")
except Exception as e:
    _fail("Tavily", e)

## Step 5 — Verify LangSmith

A call to `list_projects` confirms the API key authenticates against the configured endpoint. This is a low-cost probe that fails fast on auth or connectivity errors.


In [ ]:
from langsmith import Client

try:
    ls_client = Client()
    projects = list(ls_client.list_projects(limit=1))
    _ok("LangSmith auth", f"endpoint reachable ({endpoint})")
    if projects:
        _ok("LangSmith workspace", "existing projects found")
    else:
        _ok("LangSmith workspace", "no projects yet — a later module will create one")
except Exception as e:
    _fail("LangSmith", e)

## Next Steps

If every verification above succeeded, the environment is configured.

- Engineers new to the `deepagents` framework: open `01_build_a_deep_agent_optional.ipynb`.
- Engineers already familiar with `deepagents`: open `02_tracing.ipynb`.

Both paths converge at Module 02.


## Troubleshooting

<details>
<summary><strong>Model provider: "Missing credentials" or 401</strong></summary>

Verify the API key for the configured provider is set in `.env`. By default that's `ANTHROPIC_API_KEY`. For OpenAI, Azure, or Bedrock, set the corresponding variables in `.env` and uncomment the matching initializer in `utils/models.py`.
</details>

<details>
<summary><strong>LangSmith: connection refused or SSL errors (self-hosted)</strong></summary>

VPN may be required, or `LANGSMITH_ENDPOINT` may be misconfigured. Confirm the URL with the operator of your control plane. The default cloud endpoint is `https://api.smith.langchain.com`.
</details>

<details>
<summary><strong>Tavily fell back to canned content</strong></summary>

`utils/search.py` returns a canned response when Tavily fails — the prefix `[fallback content; ...]` is the indicator. Either `TAVILY_API_KEY` is invalid or Tavily is briefly unavailable. Verify the key first.
</details>

<details>
<summary><strong>"No module named 'utils'" or 'agents'</strong></summary>

The Step 1 cell prepends the repo root to `sys.path`. If this notebook has been moved, update the `Path().resolve().parent` line to point at the repo root.
</details>
